Para emepezar, poner el jupyter en vs code con el kernel de python

Si no os sale, ejecutad estos tres comandos a la altura del path T20
- python3 -m venv .venv
- source .venv/bin/activate
- python -m pip install ipykernel

Sección	Contenido	Responsable
0. Introducción	Objetivo, integrantes y librerías	JORGE
1. Reparto	Apartado a): generación de participaciones	JORGE
2. Recuperación	Apartado b): interpolación de Newton	ARJO
3. Comprobación	Apartado c): ejemplo del profesor	Compañero INIESTA
4. Umbral	Apartado d): pruebas y explicación matemática	Compañero INIESTA
5. Aplicación AES	Apartado e): clave AES-256	FRAN
6. Lagrange	Apartado f): recuperación alternativa	FRAN (OPCIONAL)
7. Conclusiones	Resultados y comprobaciones finales	TODOS

Cada sección tendrá una explicación breve, su código Python comentado y los resultados de ejecución.
-
-
-
-
-
-
-
-
-
-
-
-


# Seguridad y Protección de Sistemas Informáticos
## Trabajo de Python — Esquema de Shamir

**Curso:** 2026/2027

**Integrantes:**
- Jorge Castillo Fuentes:
- Alfredo Iniesta Garcia:
- Francisco Pradillo Martinez:
- Alvaro Arjona Jimenez:

## Objetivo

Implementar el esquema de Shamir para compartir secretos,
permitiendo repartir un secreto entre varios participantes
y recuperarlo cuando se dispone de suficientes participaciones.

## Apartados

1. Reparto del secreto
2. Recuperación mediante interpolación de Newton
3. Comprobación con el ejemplo de los apuntes
4. Comprobación del umbral de seguridad
5. Aplicación con una clave AES de 256 bits
6. Interpolación de Lagrange (opcional)

## 1. Reparto del secreto

En este apartado se implementará una función que permita
dividir un secreto en n participaciones mediante el
esquema de Shamir.

Para recuperar el secreto se necesitarán al menos
t participaciones.

La función recibirá:
- s: secreto que queremos repartir.
- n: número de participantes.
- t: número mínimo de participaciones.
- bits: tamaño máximo del secreto en bits.

La función devolverá el número primo p y las
participaciones generadas.

In [12]:
import secrets
import subprocess
from itertools import combinations
import time

## 1. Reparto del secreto

En este apartado implementamos el reparto de un secreto
mediante el esquema de Shamir.

La función recibe el secreto, el número de participantes,
el umbral mínimo para recuperarlo y el tamaño máximo
del secreto en bits.

Primero comprobamos los parámetros. Después generamos
un número primo y los coeficientes aleatorios del
polinomio. Por último, elegimos los puntos y calculamos
las participaciones de cada participante.

Todas las operaciones del polinomio se realizan módulo p.

In [13]:
def validar_parametros(s, n, t, bits):
    # Comprobamos las condiciones del enunciado.
    if not all(type(v) is int for v in (s, n, t, bits)):
        raise TypeError("Los parámetros deben ser enteros")

    if n < 3:
        raise ValueError("Debe haber al menos 3 participantes")

    if t < 2 or t > n:
        raise ValueError("El umbral debe estar entre 2 y n")

    if bits < 1:
        raise ValueError("El tamaño en bits debe ser positivo")

    if s < 0 or s >= 2**bits:
        raise ValueError("El secreto no cabe en los bits indicados")


def generar_primo(bits, n):
    # El primo debe superar tanto a n como a cualquier
    # secreto que pueda representarse con esos bits.
    bits_primo = max(bits + 1, n.bit_length() + 1)

    resultado = subprocess.run(
        ["openssl", "prime", "-generate", "-bits", str(bits_primo)],
        capture_output=True,
        text=True,
        check=True
    )

    return int(resultado.stdout.strip())


def evaluar_polinomio(coeficientes, x, p):
    # Evaluamos el polinomio módulo p mediante Horner.
    resultado = 0

    for coeficiente in reversed(coeficientes):
        resultado = (resultado * x + coeficiente) % p

    return resultado


def repartir_secreto(s, n, t, bits):
    validar_parametros(s, n, t, bits)

    # Generamos el primo.
    p = generar_primo(bits, n)

    # El primer coeficiente es el secreto.
    # Los demás se generan aleatoriamente, incluido el cero.
    coeficientes = [s]
    for _ in range(t - 1):
        coeficientes.append(secrets.randbelow(p))

    # Elegimos n puntos diferentes y distintos de cero.
    puntos = set()

    while len(puntos) < n:
        x = secrets.randbelow(p - 1) + 1
        puntos.add(x)

    # Calculamos la participación correspondiente a cada x.
    participaciones = []

    for x in sorted(puntos):
        y = evaluar_polinomio(coeficientes, x, p)
        participaciones.append((x, y))

    return p, participaciones

In [14]:
# Ejemplo con 5 participantes y umbral de 3.
s = 1234
n = 5
t = 3
bits = 16

p, participaciones = repartir_secreto(s, n, t, bits)

print("Secreto:", s)
print("Número primo:", p)
print("Participantes:", n)
print("Umbral:", t)

print("\nParticipaciones generadas:")
for i, (x, y) in enumerate(participaciones, start=1):
    print(f"Participante {i}: ({x}, {y})")

print("\nComprobaciones:")
print("Primo mayor que n:", p > n)
print("Primo mayor que todos los secretos posibles:", p >= 2**bits)
print("Número correcto de participaciones:", len(participaciones) == n)
print("Puntos distintos:", len({x for x, y in participaciones}) == n)
print("Puntos no nulos:", all(x != 0 for x, y in participaciones))

Secreto: 1234
Número primo: 101573
Participantes: 5
Umbral: 3

Participaciones generadas:
Participante 1: (7757, 43589)
Participante 2: (26764, 15261)
Participante 3: (37562, 96312)
Participante 4: (49387, 3900)
Participante 5: (101172, 70329)

Comprobaciones:
Primo mayor que n: True
Primo mayor que todos los secretos posibles: True
Número correcto de participaciones: True
Puntos distintos: True
Puntos no nulos: True
